# Magi baseline prediction cache

Select the **Godfather** kernel and run top to bottom. This notebook processes every development sequence with frozen MagiV2 **on CUDA**, then matches raw Magi OCR to GT dialogue. It does not run Paddle, remove characters, cluster E entities, assign GT identities to characters, or train a projection.

Original text-box order and raw model outputs are preserved. Matching may group two adjacent bubbles for evaluation, but never rewrites the raw prediction. GT reuse is allowed. A high fuzzy score is a text-alignment signal, not proof of the associated character's identity.

This notebook is independent of `FAFO.ipynb` and `pipeline1.ipynb`.

## 1. Imports and settings

<details>
<summary>Details</summary>

Torch imports first. Magi runs directly in this kernel. CUDA is required; there is no CPU model fallback. Fuzzy text comparisons run on CPU. Set `SEQUENCE_LIMIT` for a small trial, or leave it as `None` for all sequences.

</details>

In [1]:
import torch
import gc
import hashlib
import json
import unicodedata
from datetime import datetime, timezone
from difflib import SequenceMatcher
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from transformers import AutoModel

assert torch.cuda.is_available(), 'Select Godfather with CUDA-enabled PyTorch.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(DEVICE))

MAGI_ROOT = Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi')
MODEL_PATH = Path('C:/MODELS/magiv2')
SPLIT = 'development'
IMAGE_ROOT = MAGI_ROOT / 'TaskData' / SPLIT / 'images'
LABELS_PATH = MAGI_ROOT / 'TaskData' / SPLIT / 'labels.jsonl'
CACHE_ROOT = MAGI_ROOT / 'paddleOCR-Magi' / 'cache_baseline' / SPLIT
SEQUENCE_LIMIT = None
FORCE_RAW = False
FORCE_MATCHES = False
MODEL_REVISION = 'local-magiv2-1'  # Bump if replacing weights without metadata changes.
RAW_VERSION = 1
MATCH_VERSION = 1  # Bump when changing matching code.

GT_MATCH_SETTINGS = dict(
    strict_threshold=95.0, fallback_threshold=75.0,
    fallback_weights=dict(partial_ratio=0.40, token_overlap=0.30, ngram_overlap=0.30),
    ngram_size=3, fallback_min_chars=3, allow_gt_reuse=True,
    scoring='difflib_full_and_window_partial_v1',
    normalization='nfkc_casefold_alnum_preserve_symbol_only_v2',
)

GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Find sequences and labels

<details>
<summary>Details</summary>

Folders define the sequences to process. GT dialogue labels are loaded only for text matching, never passed to Magi. Every matching cache records the exact label hash. Missing labels do not prevent raw prediction caching; they are reported as matching failures.

</details>

In [2]:
assert IMAGE_ROOT.is_dir(), f'Missing images: {IMAGE_ROOT}'
assert MODEL_PATH.is_dir(), f'Missing model: {MODEL_PATH}'
sequence_folders = sorted(path for path in IMAGE_ROOT.iterdir() if path.is_dir())
if SEQUENCE_LIMIT is not None:
    assert SEQUENCE_LIMIT > 0
    sequence_folders = sequence_folders[:SEQUENCE_LIMIT]
assert sequence_folders, 'No sequence folders found'

labels_by_sequence = {}
with LABELS_PATH.open(encoding='utf-8') as file:
    for line in file:
        if line.strip():
            label = json.loads(line)
            assert label['sequence_id'] not in labels_by_sequence, 'Duplicate sequence label'
            labels_by_sequence[label['sequence_id']] = label
print('Sequences to process:', len(sequence_folders))
print('Cache root:', CACHE_ROOT)

Sequences to process: 80
Cache root: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_baseline\development


## 3. Small cache helpers

<details>
<summary>Details</summary>

Each stage uses a signature and data checksum. Raw signatures include page content, preprocessing, model metadata, and package versions. Matching signatures include raw output, GT labels, and matching settings. Writes are atomic; existing valid files are reused. One notebook should write this cache root at a time.

</details>

In [3]:
def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def digest(data):
    text = json.dumps(json_ready(data), sort_keys=True, ensure_ascii=True, allow_nan=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def file_hash(path):
    with path.open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(json_ready(data), indent=2, ensure_ascii=True,
                                    allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def cached_stage(folder, name, settings, compute, force=False):
    signature = digest(settings)
    path = folder / f'{name}.json'
    if path.exists() and not force:
        try:
            saved = json.loads(path.read_text(encoding='utf-8'))
            if saved['signature'] == signature and digest(saved['data']) == saved['data_hash']:
                return saved, 'cache'
        except (ValueError, KeyError, OSError):
            pass
    data = json_ready(compute())
    saved = dict(signature=signature, data_hash=digest(data),
                 created_at=datetime.now(timezone.utc).isoformat(), settings=settings, data=data)
    write_json(path, saved)
    return saved, 'computed'


model_files = []
for path in sorted(MODEL_PATH.rglob('*')):
    if path.is_file() and path.suffix in {'.json', '.py', '.safetensors', '.bin', '.pt', '.pth'}:
        stat = path.stat()
        model_files.append(dict(name=str(path.relative_to(MODEL_PATH)), size=stat.st_size,
            mtime_ns=stat.st_mtime_ns, sha256=file_hash(path) if path.suffix in {'.json', '.py'} else None))
MODEL_INFO = dict(path=str(MODEL_PATH), revision=MODEL_REVISION, files=model_files)
PACKAGES = {name: version(name) for name in ['torch', 'torchvision', 'transformers', 'numpy', 'pillow']}

## 4. Frozen Magi on GPU

<details>
<summary>Details</summary>

Load the model once, on the first raw cache miss. Keep it in evaluation mode and disable gradients. Pages use the same grayscale-to-RGB preprocessing as pipeline1. No GT, character bank, Paddle selection, or cleanup is passed into prediction. Character crop embeddings are not separately cached in this baseline notebook.

</details>

In [4]:
magi_model = None


def get_magi():
    global magi_model
    if magi_model is None:
        magi_model = AutoModel.from_pretrained(str(MODEL_PATH), trust_remote_code=True,
            local_files_only=True).to(DEVICE).eval()
        magi_model.requires_grad_(False)
        assert next(magi_model.parameters()).is_cuda, 'Magi must be on GPU'
    return magi_model


def predict_sequence(page_files):
    pages = []
    for path in page_files:
        with Image.open(path) as image:
            pages.append(np.array(image.convert('L').convert('RGB')))
    with torch.inference_mode():
        results = get_magi().do_chapter_wide_prediction(
            pages, {'images': [], 'names': []}, use_tqdm=True, do_ocr=True)
    results = json_ready(results)
    assert len(results) == len(page_files), 'Magi page count mismatch'
    for page in results:
        assert len(page['texts']) == len(page['ocr']), 'OCR/text count mismatch'
    return results

## 5. OCR matching scores

<details>
<summary>Details</summary>

Use the same normalized full-string score and fallback scores as pipeline1. Punctuation-only dialogue such as `!` remains matchable. These fuzzy scores are similarity measures, not confidence probabilities.

</details>

In [5]:
def normalize_gt_text(text):
    text = unicodedata.normalize('NFKC', text or '').casefold()
    words = ' '.join(''.join(char if char.isalnum() else ' ' for char in text).split())
    if words:
        return words
    # Punctuation/symbol-only dialogue is meaningful, not empty OCR.
    symbols = ''.join(char for char in text
                      if char.isspace() or unicodedata.category(char)[0] in {'P', 'S'})
    return ' '.join(symbols.split())


def full_fuzzy_score(a, b):
    if not a or not b:
        return 0.0
    return 100.0 * SequenceMatcher(None, a, b, autojunk=False).ratio()


def partial_fuzzy_score(a, b):
    if not a or not b:
        return 0.0
    short, long = (a, b) if len(a) <= len(b) else (b, a)
    if short in long:
        return 100.0
    return max(full_fuzzy_score(short, long[start:start+len(short)])
               for start in range(len(long)-len(short)+1))


def overlap_score(a, b):
    if not a or not b:
        return 0.0
    return 100.0 * len(a & b) / min(len(a), len(b))


def text_ngrams(text, size):
    return {text[i:i+size] for i in range(max(0, len(text)-size+1))}


def gt_scores(ocr_text, gt_text, settings):
    a, b = normalize_gt_text(ocr_text), normalize_gt_text(gt_text)
    scores = dict(full_ratio=full_fuzzy_score(a, b), partial_ratio=partial_fuzzy_score(a, b),
                  token_overlap=overlap_score(set(a.split()), set(b.split())),
                  ngram_overlap=overlap_score(text_ngrams(a, settings['ngram_size']),
                                              text_ngrams(b, settings['ngram_size'])))
    scores['fallback_score'] = sum(settings['fallback_weights'][key] * scores[key]
                                   for key in settings['fallback_weights'])
    return scores


def best_gt_candidate(text, gt_lines, settings, score_key):
    best = None
    for gt_index, gt in enumerate(gt_lines):
        if not normalize_gt_text(gt['text']):
            continue
        scores = gt_scores(text, gt['text'], settings)
        if best is None or scores[score_key] > best['scores'][score_key]:
            best = dict(gt_index=gt_index, gt_text=gt['text'], gt_speaker=gt['speaker'], scores=scores)
    return best

## 6. Match raw OCR to GT

<details>
<summary>Details</summary>

Strict single matches come first, then consecutive unresolved pairs on the same page, then tentative fallback matches. Multiple independent bubbles can match the same GT line. Store one bubble row per original text index, including empty OCR. GT speaker is saved as label metadata only; no character identity is assigned. `eligible_for_training` means a direct/joined full score of at least 95%; fallback is excluded. Ambiguous/repeated GT text and bubble-character links must still be reviewed before identity supervision.

</details>

In [6]:
def match_ocr_to_gt(results, labels, settings):
    assert len(results) == len(labels['pages']), 'GT/page count mismatch'
    assert settings['allow_gt_reuse'], 'This performance stage allows repeated GT matches'
    assert 0 <= settings['strict_threshold'] <= 100 and 0 <= settings['fallback_threshold'] <= 100
    assert settings['ngram_size'] >= 1 and settings['fallback_min_chars'] >= 0
    weights = settings['fallback_weights']
    assert set(weights) == {'partial_ratio', 'token_overlap', 'ngram_overlap'}
    assert all(weight >= 0 for weight in weights.values()) and abs(sum(weights.values())-1) < 1e-9
    groups, bubble_matches, gt_coverage = [], [], []

    for page_index, result in enumerate(results):
        texts, gt_lines = result['ocr'], labels['pages'][page_index]
        assert len(texts) == len(result['texts']), 'OCR/box count mismatch'
        assignments = {}

        def add_group(indices, stage, candidate, reason):
            text = ' '.join(texts[index] for index in indices)
            accepted = stage != 'unmatched'
            group = dict(sequence_id=labels['sequence_id'], page_index=page_index,
                         text_indices=list(indices), bubble_ids=[f'PG{page_index+1}:T{i+1}' for i in indices],
                         ocr_text=text, normalized_ocr=normalize_gt_text(text), stage=stage, reason=reason,
                         gt_index=candidate['gt_index'] if accepted else None,
                         gt_text=candidate['gt_text'] if accepted else None,
                         gt_speaker=candidate['gt_speaker'] if accepted else None,
                         scores=candidate['scores'] if candidate else None,
                         best_candidate=candidate, needs_review=stage in {'fallback', 'unmatched'})
            for index in indices:
                assert index not in assignments, 'Bubble was matched twice'
                assignments[index] = group
            return group

        # Stage 1: each bubble gets its own best match. GT lines are never reserved.
        for text_index, text in enumerate(texts):
            if normalize_gt_text(text):
                candidate = best_gt_candidate(text, gt_lines, settings, 'full_ratio')
                if candidate and candidate['scores']['full_ratio'] >= settings['strict_threshold']:
                    add_group([text_index], 'direct', candidate, 'strict_single')

        # Stage 2: only genuinely consecutive unresolved pairs, never across pages.
        joined_candidates = []
        for text_index in range(len(texts)-1):
            indices = [text_index, text_index+1]
            if any(index in assignments for index in indices):
                continue
            if not all(normalize_gt_text(texts[index]) for index in indices):
                continue
            joined_text = ' '.join(texts[index] for index in indices)
            candidate = best_gt_candidate(joined_text, gt_lines, settings, 'full_ratio')
            if candidate and candidate['scores']['full_ratio'] >= settings['strict_threshold']:
                joined_candidates.append((indices, candidate))
        joined_candidates.sort(key=lambda item: item[1]['scores']['full_ratio'], reverse=True)
        for indices, candidate in joined_candidates:
            if not any(index in assignments for index in indices):
                add_group(indices, 'adjacent_join', candidate, 'strict_adjacent_pair')

        # Stage 3: single remaining fragments; tentative matches always need review.
        for text_index, text in enumerate(texts):
            if text_index in assignments:
                continue
            normalized = normalize_gt_text(text)
            candidate = best_gt_candidate(text, gt_lines, settings, 'fallback_score') if normalized else None
            signal = sum(char.isalnum() for char in normalized)
            if candidate and signal >= settings['fallback_min_chars'] and candidate['scores']['fallback_score'] >= settings['fallback_threshold']:
                add_group([text_index], 'fallback', candidate, 'partial_overlap_fragment')
            else:
                reason = ('empty_ocr' if not normalized else 'no_gt_lines' if not candidate else
                          'fragment_too_short' if signal < settings['fallback_min_chars'] else 'below_fallback_threshold')
                add_group([text_index], 'unmatched', candidate, reason)

        # Store groups and bubble rows in ORIGINAL order, irrespective of matching priority.
        for text_index in range(len(texts)):
            group = assignments[text_index]
            if text_index == group['text_indices'][0]:
                group['match_id'] = f'M{len(groups)+1}'
                group['eligible_for_training'] = group['stage'] in {'direct', 'adjacent_join'} and group['scores']['full_ratio'] >= settings['strict_threshold']
                groups.append(group)
            bubble_matches.append(dict(sequence_id=labels['sequence_id'], page_index=page_index,
                text_index=text_index, text_id=f'T{text_index+1}', box=result['texts'][text_index], ocr_text=texts[text_index],
                match_id=group['match_id'], group_text_indices=group['text_indices'], stage=group['stage'],
                gt_index=group['gt_index'], gt_text=group['gt_text'], gt_speaker=group['gt_speaker'],
                scores=group['scores'], reason=group['reason'], needs_review=group['needs_review'],
                eligible_for_training=group['eligible_for_training'], ocr_source='magi'))
        for gt_index, gt in enumerate(gt_lines):
            matched_groups = [g for g in groups if g['page_index'] == page_index and g['gt_index'] == gt_index]
            gt_coverage.append(dict(page_index=page_index, gt_index=gt_index, gt_text=gt['text'],
                gt_speaker=gt['speaker'], match_ids=[g['match_id'] for g in matched_groups],
                group_count=len(matched_groups), bubble_count=sum(len(g['text_indices']) for g in matched_groups),
                strict_group_count=sum(g['stage'] in {'direct', 'adjacent_join'} for g in matched_groups)))
    return dict(settings=settings, groups=groups, bubble_matches=bubble_matches, gt_coverage=gt_coverage)

## 7. Process every sequence

<details>
<summary>Details</summary>

Run this cell to perform inference. Progress prints once per sequence plus Magi progress. A failed sequence is recorded and the batch continues. Completed raw caches remain usable even if GT matching fails. `batch_report.json` is updated after every sequence; interrupting the batch does not discard completed caches. Rerun to resume. Existing invalidated files are recomputed, not deleted.

</details>

In [7]:
report = []
try:
    for number, sequence_folder in enumerate(sequence_folders, start=1):
        sequence_id = sequence_folder.name
        folder = CACHE_ROOT / sequence_id
        row = dict(sequence_id=sequence_id, status='running', raw_status=None,
                   match_status=None, error=None)
        print(f'[{number}/{len(sequence_folders)}] {sequence_id}', flush=True)
        try:
            page_files = sorted(sequence_folder.glob('*.png'))
            assert page_files, 'No PNG pages found'
            page_metadata = [dict(name=path.name, sha256=file_hash(path)) for path in page_files]
            raw_settings = dict(sequence_id=sequence_id, split=SPLIT, pages=page_metadata,
                preprocessing='grayscale_then_RGB', model=MODEL_INFO, packages=PACKAGES,
                do_ocr=True, algorithm_version=RAW_VERSION)
            raw_cache, row['raw_status'] = cached_stage(folder, 'magi_raw', raw_settings,
                lambda: predict_sequence(page_files), force=FORCE_RAW)
            raw = raw_cache['data']
            row['pages'] = len(raw)
            row['bubbles'] = sum(len(page['texts']) for page in raw)
            # Record raw provenance before matching, even if matching later fails.
            sequence_manifest = dict(sequence_id=sequence_id, split=SPLIT, pages=page_metadata,
                magi_raw=dict(file='magi_raw.json', signature=raw_cache['signature'],
                              data_hash=raw_cache['data_hash']))
            write_json(folder / 'manifest.json', sequence_manifest)
            label = labels_by_sequence.get(sequence_id)
            assert label is not None, f'Missing GT labels: {sequence_id}'
            assert len(label['pages']) == len(page_files), 'GT/page count mismatch'
            match_settings = dict(algorithm_version=MATCH_VERSION,
                magi_signature=raw_cache['signature'], magi_data_hash=raw_cache['data_hash'],
                labels_sha256=digest(label), matching=GT_MATCH_SETTINGS)
            match_cache, row['match_status'] = cached_stage(folder, 'gt_matches', match_settings,
                lambda: match_ocr_to_gt(raw, label, GT_MATCH_SETTINGS), force=FORCE_MATCHES)
            matches = match_cache['data']
            row['eligible_bubbles'] = sum(item['eligible_for_training'] for item in matches['bubble_matches'])
            row['unmatched_bubbles'] = sum(item['stage'] == 'unmatched' for item in matches['bubble_matches'])
            sequence_manifest['gt_matches'] = dict(file='gt_matches.json',
                signature=match_cache['signature'], data_hash=match_cache['data_hash'],
                labels_sha256=digest(label))
            write_json(folder / 'manifest.json', sequence_manifest)
            row['status'] = 'ok'
            print(f"  raw={row['raw_status']} | matching={row['match_status']} | bubbles={row['bubbles']} | eligible={row['eligible_bubbles']}", flush=True)
        except Exception as error:
            row['status'] = 'error'
            row['error'] = f'{type(error).__name__}: {error}'
            print('  ERROR:', row['error'], flush=True)
        report.append(row)
        write_json(CACHE_ROOT / 'batch_report.json', dict(split=SPLIT,
            updated_at=datetime.now(timezone.utc).isoformat(), sequences=report))
        gc.collect()
        torch.cuda.empty_cache()
finally:
    magi_model = None
    gc.collect()
    torch.cuda.empty_cache()

display(pd.DataFrame(report))
print('Successful:', sum(row['status'] == 'ok' for row in report), '/', len(report))

[1/80] seq_032cbb4dc8686963


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
c:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\torch\nn\modules\module.py:2593: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
c:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\torch\nn\modules\batchnorm.py:141: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=Tr

  raw=computed | matching=computed | bubbles=34 | eligible=29


[2/80] seq_06ec0acde37dffa4


100%|██████████| 2/2 [00:06<00:00,  3.38s/it]


...

  raw=computed | matching=computed | bubbles=44 | eligible=37
[80/80] seq_fc4c2b84b6ec32ff


100%|██████████| 1/1 [00:02<00:00,  2.92s/it]

  raw=computed | matching=computed | bubbles=22 | eligible=19


,sequence_id,status,raw_status,match_status,error,pages,bubbles,eligible_bubbles,unmatched_bubbles
0,seq_032cbb4dc8686963,ok,computed,computed,None,3,34,29,5
1,seq_06ec0acde37dffa4,ok,computed,computed,None,3,43,23,16
2,seq_07838fb26cd1be54,ok,computed,computed,None,3,18,16,2
3,seq_098e260f231aa789,ok,computed,computed,None,3,32,30,2
4,seq_0dc2e3d874c804fc,ok,computed,computed,None,3,28,24,2
...,...,...,...,...,...,...,...,...,...
75,seq_f9a4e67ae3c018f6,ok,computed,computed,None,3,31,24,5
76,seq_fa2eee6269a5dad3,ok,computed,computed,None,3,27,20,6
77,seq_fa5b05d0f22bff15,ok,computed,computed,None,3,38,32,6
78,seq_fb6d15ad78b7191c,ok,computed,computed,None,3,44,37,5


Successful: 80 / 80


## 8. Inspect a cached sequence

<details>
<summary>Details</summary>

This only reads caches. Raw pages retain all text boxes and original reading order. Joined match groups do not merge the raw bubbles. GT coverage describes text alignment; no speaker-identity accuracy is claimed.

</details>

In [8]:
INSPECT_SEQUENCE = sequence_folders[0].name
folder = CACHE_ROOT / INSPECT_SEQUENCE
raw = json.loads((folder / 'magi_raw.json').read_text(encoding='utf-8'))['data']
matches = json.loads((folder / 'gt_matches.json').read_text(encoding='utf-8'))['data']
display(pd.DataFrame(matches['bubble_matches'])[
    ['page_index', 'text_id', 'ocr_text', 'stage', 'gt_index', 'gt_text',
     'gt_speaker', 'scores', 'eligible_for_training']])
print('Raw page fields:', list(raw[0]))
print('Bubble counts:', [len(page['texts']) for page in raw])

,page_index,text_id,ocr_text,stage,gt_index,gt_text,gt_speaker,scores,eligible_for_training
0,0,T1,I mean...,direct,0.0,I MEAN...,character1,"{'full_ratio': 100.0, 'partial_ratio': 100.0, ...",True
1,0,T2,You're the only one,direct,1.0,YOU'RE THE ONLY ONE,character1,"{'full_ratio': 100.0, 'partial_ratio': 100.0, ...",True
2,0,T3,who's ever seen me like this.,direct,2.0,WHO'S EVER SEEN ME LIKE THIS.,character1,"{'full_ratio': 100.0, 'partial_ratio': 100.0, ...",True
3,0,T4,TH...,direct,3.0,TH...,character2,"{'full_ratio': 100.0, 'partial_ratio': 100.0, ...",True
4,0,T5,THE ONLY ONE...?,direct,4.0,THE ONLY ONE...?,character2,"{'full_ratio': 100.0, 'partial_ratio': 100.0, ...",True
5,0,T6,So I can't ask anyone else.,direct,5.0,SO I CAN'T ASK ANYONE ELSE.,character1,"{'full_ratio': 100.0, 'partial_ratio': 100.0, ...",True
6,0,T7,IT HAS TO BE YOU...,direct,7.0,IT HAS TO BE YOU...,character1,"{'full_ratio': 100.0, 'partial_ratio': 100.0, ...",True
7,0,T8,I...,direct,6.0,I...,character1,"{'full_ratio': 100.0, 'partial_ratio': 100.0, ...",True
8,0,T9,UM...,direct,8.0,UM...,character2,"{'full_ratio': 100.0, 'partial_ratio': 100.0, ...",True
9,0,T10,ER...,direct,9.0,ER...,character2,"{'full_ratio': 100.0, 'partial_ratio': 100.0, ...",True


Raw page fields: ['panels', 'texts', 'characters', 'tails', 'text_character_associations', 'text_tail_associations', 'character_cluster_labels', 'is_essential_text', 'ocr', 'character_names']
Bubble counts: [12, 12, 10]


## Files saved

For each sequence under `cache_baseline/development/<sequence_id>/`:

- **`magi_raw.json`**: cache wrapper plus unchanged JSON-compatible Magi page results, including boxes, raw OCR, associations, cluster labels, and essential flags. Original page/text/character indices remain intact.
- **`gt_matches.json`**: cache wrapper plus matching settings, match groups, one row per original bubble, and GT coverage. Each group and bubble records its stage, scores, GT reference, review flag, and text-score eligibility for later training.
- **`manifest.json`**: ordered page filenames and hashes, plus raw/matching signatures and checksums.

`cache_baseline/development/batch_report.json` records sequence outcomes for the latest run. Cache wrappers contain `signature`, `data_hash`, `created_at`, `settings`, and `data`. Labels affect matching only. No GT-speaker to bubble-speaker mapping is performed here.

## 9. Training-review match summary

<details>
<summary>What these counts mean</summary>

Summarize successfully processed sequences from this batch. Qualifying bubbles have a direct or adjacent-joined full-string match of at least 95%. A joined pair contributes two bubbles but one match group. Multiple bubbles may match the same GT dialogue.

For each qualifying bubble, count distinct character indices in raw Magi text-character associations: exactly one, multiple, or none. “With a character association” includes both single and multiple links. These are predicted links, not verified GT speaker identities. No speaker matching or new inference runs here.

Validate cache checksums and matching/raw provenance before counting. Failed sequences or inconsistent caches are reported and excluded.

</details>


In [9]:
def count_review_matches(raw_pages, matches):
    links = {}
    for page_index, page in enumerate(raw_pages):
        for text_index, character_index in page['text_character_associations']:
            assert 0 <= text_index < len(page['texts']), 'Invalid text link'
            assert 0 <= character_index < len(page['characters']), 'Invalid character link'
            links.setdefault((page_index, text_index), set()).add(character_index)

    counts = dict(eligible_bubbles=0, eligible_match_groups=0,
                  with_character_association=0, exactly_one_character=0,
                  multiple_characters=0, no_character=0)
    group_ids = set()
    expected_order = [(p, t) for p, page in enumerate(raw_pages)
                      for t in range(len(page['texts']))]
    assert [(r['page_index'], r['text_index']) for r in matches['bubble_matches']] == expected_order
    for row in matches['bubble_matches']:
        page_index, text_index = row['page_index'], row['text_index']
        assert row['ocr_text'] == raw_pages[page_index]['ocr'][text_index], 'OCR alignment changed'
        assert row['box'] == raw_pages[page_index]['texts'][text_index], 'Box alignment changed'
        if not (row['eligible_for_training'] and row['stage'] in {'direct', 'adjacent_join'}
                and row['scores']['full_ratio'] >= 95.0):
            continue
        counts['eligible_bubbles'] += 1
        group_ids.add(row['match_id'])
        linked_characters = len(links.get((page_index, text_index), set()))
        if linked_characters:
            counts['with_character_association'] += 1
        category = ('no_character' if linked_characters == 0 else
                    'exactly_one_character' if linked_characters == 1 else 'multiple_characters')
        counts[category] += 1
    counts['eligible_match_groups'] = len(group_ids)
    return counts


summary_rows, summary_errors = [], []
# This also works after reopening the notebook and loading its setup/helper cells.
batch_rows = report if 'report' in globals() else json.loads(
    (CACHE_ROOT / 'batch_report.json').read_text(encoding='utf-8'))['sequences']
for batch_row in batch_rows:
    sequence_id = batch_row['sequence_id']
    if batch_row['status'] != 'ok':
        summary_errors.append(dict(sequence_id=sequence_id, reason=batch_row['error']))
        continue
    try:
        folder = CACHE_ROOT / sequence_id
        raw_cache = json.loads((folder / 'magi_raw.json').read_text(encoding='utf-8'))
        match_cache = json.loads((folder / 'gt_matches.json').read_text(encoding='utf-8'))
        assert digest(raw_cache['data']) == raw_cache['data_hash'], 'Raw cache checksum mismatch'
        assert digest(match_cache['data']) == match_cache['data_hash'], 'Match cache checksum mismatch'
        assert match_cache['settings']['magi_signature'] == raw_cache['signature'], 'Raw signature mismatch'
        assert match_cache['settings']['magi_data_hash'] == raw_cache['data_hash'], 'Raw data mismatch'
        summary_rows.append(dict(sequence_id=sequence_id,
            **count_review_matches(raw_cache['data'], match_cache['data'])))
    except Exception as error:
        summary_errors.append(dict(sequence_id=sequence_id, reason=str(error)))

summary_columns = ['eligible_bubbles', 'eligible_match_groups', 'with_character_association',
                   'exactly_one_character', 'multiple_characters', 'no_character']
totals = {key: sum(row[key] for row in summary_rows) for key in summary_columns}
assert totals['eligible_bubbles'] == (totals['exactly_one_character'] +
    totals['multiple_characters'] + totals['no_character'])
percentage = (100 * totals['with_character_association'] / totals['eligible_bubbles']
              if totals['eligible_bubbles'] else 0.0)
print(f'Sequences summarized: {len(summary_rows)} | excluded: {len(summary_errors)}')
print(f"Training-review matches: {totals['eligible_bubbles']} bubbles in {totals['eligible_match_groups']} match groups")
print(f"With a Magi character association: {totals['with_character_association']} ({percentage:.1f}%)")
print(f"  Exactly one character: {totals['exactly_one_character']}")
print(f"  Multiple characters:   {totals['multiple_characters']}")
print(f"  No character:          {totals['no_character']}")
display(pd.DataFrame(summary_rows, columns=['sequence_id'] + summary_columns))
if summary_errors:
    print('Excluded sequences:')
    display(pd.DataFrame(summary_errors))


Sequences summarized: 80 | excluded: 0
Training-review matches: 1654 bubbles in 1601 match groups
With a Magi character association: 1598 (96.6%)
  Exactly one character: 1598
  Multiple characters:   0
  No character:          56


,sequence_id,eligible_bubbles,eligible_match_groups,with_character_association,exactly_one_character,multiple_characters,no_character
0,seq_032cbb4dc8686963,29,28,29,29,0,0
1,seq_06ec0acde37dffa4,23,18,23,23,0,0
2,seq_07838fb26cd1be54,16,16,16,16,0,0
3,seq_098e260f231aa789,30,30,28,28,0,2
4,seq_0dc2e3d874c804fc,24,24,23,23,0,1
...,...,...,...,...,...,...,...
75,seq_f9a4e67ae3c018f6,24,24,23,23,0,1
76,seq_fa2eee6269a5dad3,20,20,20,20,0,0
77,seq_fa5b05d0f22bff15,32,32,31,31,0,1
78,seq_fb6d15ad78b7191c,37,37,34,34,0,3
